In [ ]:
# 📦 STEP 1: Import All Required Libraries
import os
import time
import warnings
from pathlib import Path
from typing import Dict, List, Any

# PDF processing
import fitz  # PyMuPDF

# LangChain components
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq
from langchain_core.documents import Document

# Pinecone
from pinecone import Pinecone

# LangGraph for workflow
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# Environment variables
from dotenv import load_dotenv

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

In [ ]:
# 🔑 STEP 2: Setup Environment and API Keys
load_dotenv()

# API Keys (make sure these are in your .env file or set them directly)
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Verify API keys
print("🔑 API Keys Status:")
print(f"   Cohere: {'✅' if COHERE_API_KEY else '❌'}")
print(f"   Pinecone: {'✅' if PINECONE_API_KEY else '❌'}")
print(f"   Groq: {'✅' if GROQ_API_KEY else '❌'}")

print("\n🔧 Environment setup completed!")

In [ ]:
# 🧠 STEP 3: Initialize Cohere Embeddings
embeddings = CohereEmbeddings(
    cohere_api_key=COHERE_API_KEY,
    model="embed-english-v3.0"
)

# Test embeddings
try:
    test_embedding = embeddings.embed_query("plant disease symptoms")
    print(f"✅ Cohere embeddings working! Dimension: {len(test_embedding)}")
except Exception as e:
    print(f"❌ Embeddings error: {e}")

In [ ]:
# 🗄️ STEP 4: Setup Pinecone Vector Store
pc = Pinecone(api_key=PINECONE_API_KEY)

# Connect to your existing index
index_name = "hi"  # Your existing index name
index = pc.Index(index_name)

# Create vector store
vector_store = PineconeVectorStore(
    embedding=embeddings,
    index=index
)

# Test connection
try:
    stats = index.describe_index_stats()
    print(f"✅ Pinecone connected!")
    print(f"   • Total vectors: {stats.total_vector_count}")
    print(f"   • Dimension: {stats.dimension}")
except Exception as e:
    print(f"❌ Pinecone error: {e}")

In [ ]:
# 🤖 STEP 5: Setup Groq LLM
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
    temperature=0.1
)

# Test LLM
try:
    test_response = llm.invoke("What is plant pathology? Answer in one sentence.")
    print(f"✅ Groq LLM working!")
    print(f"   Test response: {test_response.content[:80]}...")
except Exception as e:
    print(f"❌ LLM error: {e}")

In [ ]:
# ✅ STEP 6: Final System Check
print("🔍 Final System Status:")
print(f"   Embeddings: ✅")
print(f"   Vector Store: ✅")
print(f"   LLM: ✅")
print(f"   Index: ✅")
print("\n🎉 All systems ready for PDF processing!")

In [ ]:
# 📄 STEP 7: Fixed PDF Processing
pdf_path = "potato_leaf_disease - Copy/train/leaf_train.pdf"

# Initialize text splitter for optimal chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Optimal size for plant disease content
    chunk_overlap=200,  # Overlap to maintain context
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def process_plant_disease_pdf_fixed(pdf_path: str):
    """Fixed PDF processing function"""
    
    try:
        print(f"🔍 Processing {pdf_path.split('/')[-1]}...")
        
        # Extract text from PDF - FIX: Don't close document until we're done
        doc = fitz.open(pdf_path)
        full_content = ""
        page_count = len(doc)  # Get page count before processing
        
        # Chunk per page so the page number travels as METADATA. Previously
        # every page was concatenated with an inline "--- Page N ---" marker,
        # which got embedded as part of the chunk (semantic noise) and was then
        # unrecoverable, so answers could not cite a page. Citations matter here:
        # this bot gives pesticide advice and a farmer should be able to check it.
        pages = []
        for page_num in range(page_count):
            page_text = doc[page_num].get_text()
            if page_text.strip():
                pages.append((page_num + 1, page_text))

        # NOW close the document after we are done with it
        doc.close()
        print(f"📖 Extracted text from {page_count} pages")
        
        if not full_content.strip():
            print("❌ No text content found in PDF")
            return 0
        
        # Split each page separately, keeping (page, chunk) pairs.
        page_chunks = []
        for page_no, page_text in pages:
            for chunk in text_splitter.split_text(page_text):
                if chunk.strip():
                    page_chunks.append((page_no, chunk))
        text_chunks = [c for _, c in page_chunks]
        print(f"Created {len(text_chunks)} chunks across {len(pages)} pages")
        
        if not text_chunks:
            print("❌ No text chunks created")
            return 0
        
        # Process each chunk
        vectors_to_upsert = []
        
        for i, (page_no, chunk) in enumerate(page_chunks):
            try:
                # Generate embedding
                embedding = embeddings.embed_documents([chunk])[0]
                
                # Create metadata (includes 'text' key to prevent warnings)
                metadata = {
                    "document": "leaf_train.pdf",
                    "file_path": pdf_path,
                    "file_type": "pdf",
                    "chunk_id": i,
                    "page": page_no,   # enables span-level citation
                    "total_chunks": len(text_chunks),
                    "text": chunk,  # This prevents the 'no text key' warnings
                    "content_type": "plant_disease_research",
                    "source": "leaf_train_pdf"
                }
                
                # Add to vectors list
                vectors_to_upsert.append({
                    "id": f"leaf_train_chunk_{i}",
                    "values": embedding,
                    "metadata": metadata
                })
                
                print(f"📤 Processed chunk {i+1}/{len(text_chunks)}")
                time.sleep(0.2)  # Reduced sleep time
                
            except Exception as chunk_error:
                print(f"❌ Error processing chunk {i}: {chunk_error}")
                continue
        
        # Batch upsert to Pinecone
        if vectors_to_upsert:
            try:
                index.upsert(vectors=vectors_to_upsert)
                print(f"✅ Successfully uploaded {len(vectors_to_upsert)} chunks to Pinecone!")
            except Exception as upload_error:
                print(f"❌ Error uploading to Pinecone: {upload_error}")
                return 0
        else:
            print("❌ No vectors to upload")
            return 0
            
        return len(vectors_to_upsert)
        
    except FileNotFoundError:
        print(f"❌ PDF file not found: {pdf_path}")
        print("💡 Make sure the file path is correct:")
        print(f"   Current path: {pdf_path}")
        return 0
    except Exception as e:
        print(f"❌ Error processing PDF: {e}")
        return 0

# Process your PDF with the fixed function
chunks_processed = process_plant_disease_pdf_fixed(pdf_path)
print(f"\n🎉 PDF processing completed! {chunks_processed} chunks added to knowledge base.")

In [ ]:
# 🔗 STEP 8: Setup Clean RAG System with LangGraph

# Define state for the RAG workflow
class RAGState(TypedDict):
    question: str
    documents: List[Document]
    answer: str

def retrieve_documents(state: RAGState) -> RAGState:
    """Retrieve relevant documents from vector store"""
    question = state["question"]
    
    # Search for relevant documents
    docs = vector_store.similarity_search(question, k=3)
    
    return {"question": question, "documents": docs, "answer": ""}

def generate_answer(state: RAGState) -> RAGState:
    """Generate answer using LLM and retrieved documents"""
    question = state["question"]
    documents = state["documents"]
    
    # Prepare context from documents
    context = "\n\n".join([doc.page_content for doc in documents])
    
    # Create prompt for plant disease expert
    prompt = f"""
You are an expert plant pathologist with deep knowledge of plant diseases, their symptoms, causes, and management strategies.

Based on the following research context about plant diseases, provide a comprehensive and accurate answer to the question.

Research Context:
{context}

Question: {question}

Instructions:
- Provide detailed, scientific information
- Include specific symptoms, pathogens, and management strategies when relevant
- If the information is not in the context, clearly state that
- Use proper scientific terminology

Answer:"""
    
    # Generate response
    response = llm.invoke(prompt)
    
    return {"question": question, "documents": documents, "answer": response.content}

# Create the RAG workflow graph
workflow = StateGraph(RAGState)

# Add nodes
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("generate", generate_answer)

# Add edges
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph
graph = workflow.compile()

print("✅ RAG system with LangGraph setup completed!")

In [ ]:
# 🧪 STEP 9: Clean Test Functions

def ask_plant_expert(question: str):
    """Clean function to ask plant disease questions - no warnings, clean output"""
    print(f"❓ Question: {question}")
    print("🔍 Consulting plant disease knowledge base...")
    
    try:
        # Use graph for response
        result = graph.invoke({"question": question})
        
        print(f"\n🌱 Expert Answer:")
        print(f"{result['answer']}")
        print("\n" + "=" * 80)
        
        return result['answer']
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def query_plant_diseases(question: str):
    """Simple function for querying plant diseases"""
    return ask_plant_expert(question)

def detailed_plant_query(question: str, show_sources: bool = False):
    """Advanced query with optional source information"""
    print(f"❓ Advanced Query: {question}")
    print("🔍 Analyzing plant disease database...")
    
    try:
        # Get documents for context if requested
        if show_sources:
            docs = vector_store.similarity_search(question, k=3)
            print(f"📚 Found {len(docs)} relevant sources")
        
        # Get answer
        result = graph.invoke({"question": question})
        
        print(f"\n🌱 Detailed Expert Analysis:")
        print(f"{result['answer']}")
        
        if show_sources and docs:
            print(f"\n📄 Source Preview:")
            for i, doc in enumerate(docs[:2], 1):
                print(f"   {i}. {doc.page_content[:100]}...")
        
        print("\n" + "=" * 80)
        return result['answer']
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Clean test functions ready!")

In [ ]:
# 🚀 STEP 10: Test Your Plant Disease RAG System

print("🌱 TESTING PLANT DISEASE RAG SYSTEM")
print("=" * 60)

# Test questions about plant diseases
test_questions = [
    "What are the symptoms of bacterial wilt in potato plants?",
    "How does late blight affect potato leaves?",
    "What biological control methods are effective against plant diseases?",
    "What is the accuracy of automated image-based disease detection?",
    "How do giant cells form in root-knot nematode infections?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n🧪 Test {i}:")
    ask_plant_expert(question)
    time.sleep(1)  # Rate limiting

print("\n🎉 Testing completed successfully!")

In [ ]:
# Simple query
ask_plant_expert("What are the symptoms of bacterial wilt?")

# Quick query
query_plant_diseases("How does late blight spread?")

# Advanced query with sources
detailed_plant_query("What biological controls are effective?", show_sources=True)

In [ ]:
# 💬 STEP 11: Your Interactive Plant Disease RAG System

print("💬 INTERACTIVE PLANT DISEASE RAG SYSTEM READY!")
print("=" * 60)
print("\n🎯 Available Functions:")
print("   • ask_plant_expert('your question')")
print("   • query_plant_diseases('your question')")
print("   • detailed_plant_query('your question', show_sources=True)")

print("\n🌱 Example Questions You Can Ask:")
print("   • 'What causes potato late blight?'")
print("   • 'How to control bacterial wilt?'")
print("   • 'What are the symptoms of early blight?'")
print("   • 'How effective is biological control?'")
print("   • 'What is PTGS in plant virus resistance?'")
print("   • 'What are giant cells in nematode infections?'")
print("   • 'How accurate is automated disease detection?'")

print("\n🚀 Your plant disease RAG system is ready to use!")
print("📋 System Summary:")
print("   ✅ PDF processed and uploaded to Pinecone")
print("   ✅ Clean RAG workflow with LangGraph")
print("   ✅ Expert plant pathology prompting")
print("   ✅ No warnings or confusing output")
print("   ✅ Multiple query functions available")

# Example usage (uncomment to test):
# ask_plant_expert("What are the main plant diseases discussed in this document?")
# detailed_plant_query("How does bacterial wilt spread?", show_sources=True)

In [ ]:
# Simple usage
ask_plant_expert("What are the symptoms of bacterial wilt?")

# Advanced usage with sources
detailed_plant_query("How does late blight spread?", show_sources=True)

# Quick usage
query_plant_diseases("What causes potato diseases?")